In [ ]:
import pandas as pd
from pathlib import Path
data_path = Path("../data/raw")

In [ ]:
orders = pd.read_csv(data_path / "orders.csv")
order_items = pd.read_csv(data_path / "order_items.csv")
payments = pd.read_csv(data_path / "payments.csv")
returns = pd.read_csv(data_path / "returns.csv")
customers = pd.read_csv(data_path / "customers.csv")
shipments = pd.read_csv(data_path / "shipments.csv")
products = pd.read_csv(data_path / "products.csv")

In [ ]:
order_items["line_value"] = order_items["price"] * order_items["qty"]
order_items.head()
order_totals = order_items.groupby("order_id")["line_value"].sum()
print(order_totals.head())

In [ ]:
order_totals = order_totals.reset_index() #grpby ne order id ko hee index bana diya tha, isliye reset karna pada so that wo norml column rahe when comparing with payments cuz usme normal colum hee h
order_totals.head()


In [ ]:
reconciliation = order_totals.merge(
    payments,
    on="order_id")
reconciliation.head()
reconciliation.shape


In [ ]:
print("Orders:", orders["order_id"].nunique())
print("Order totals:", order_totals["order_id"].nunique())
print("Payments:", payments["order_id"].nunique())

In [ ]:
missing_order_items = orders[
    ~orders["order_id"].isin(order_totals["order_id"])]
missing_order_items["order_id"].isin(payments["order_id"]).value_counts()  # fk validity != relationship completeness thus proved.


In [ ]:
reconciliation["difference"] = (
    reconciliation["amount"] - reconciliation["line_value"])
(reconciliation["difference"] == 0).value_counts()


ab payments ko directly galat nai bolege
the dataset may define payments.amount differently.
If a validation fails massively, question the assumption before blaming the data.
jo humne banai thi qty x price ka sum

In [ ]:
reconciliation[reconciliation["difference"] == 0]
reconciliation[["line_value", "amount"]].describe()
reconciliation[["line_value", "amount"]].corr()

In [ ]:
print("Total orders:", orders["order_id"].nunique())
print("Orders with order items:", order_totals["order_id"].nunique())
print("Orders without order items:", missing_order_items["order_id"].nunique())
print("Comparable orders:", reconciliation["order_id"].nunique())
print("Exact payment matches:", (reconciliation["difference"] == 0).sum())
print("Payment mismatches:", (reconciliation["difference"] != 0).sum())

In [ ]:
# returns and order_items
returns["order_item_id"].duplicated().sum()
return_totals = returns.groupby("order_item_id")["refund"].sum().reset_index()
return_totals.head()

In [ ]:
item_values = order_items[["order_item_id", "qty", "price"]].copy()
item_values["item_value"] = (
    item_values["qty"] * item_values["price"]
)
item_values.head()


In [ ]:
return_reconciliation = return_totals.merge(
    item_values,
    on="order_item_id"
)
return_reconciliation["difference"] = (
    return_reconciliation["refund"] - return_reconciliation["item_value"]
)
return_reconciliation.head()
(return_reconciliation["difference"] > 0).value_counts()
return_reconciliation[return_reconciliation["difference"] > 0]["difference"].describe()

In [ ]:
print("Total returned order items:", return_reconciliation["order_item_id"].nunique())
print("Refunds exceeding item value:", (return_reconciliation["difference"] > 0).sum())
print("Refunds within item value:", (return_reconciliation["difference"] <= 0).sum())
print("Average excess refund:", return_reconciliation.loc[
    return_reconciliation["difference"] > 0, "difference"
].mean()) # loc syntax is used to filter the dataframe based on a condition and select a specific column for further analysis.
print("Maximum excess refund:", return_reconciliation["difference"].max())

In [ ]:
#Customer signup date vs order date
order_customer_dates = orders[
    ["order_id", "customer_id", "order_date"]
].merge(
    customers[["customer_id", "signup_date"]],
    on="customer_id",
    how="left"
)

In [ ]:
(order_customer_dates["order_date"] < order_customer_dates["signup_date"]).sum()
order_customer_dates.loc[
    order_customer_dates["order_date"] < order_customer_dates["signup_date"],
    ["order_id", "customer_id", "order_date", "signup_date"]
].head()


In [ ]:
print("Total orders checked:", len(order_customer_dates))
print("Orders before customer signup:",(
    order_customer_dates["order_date"] < order_customer_dates["signup_date"]
).sum())
print("Orders on/after customer signup:",(
    order_customer_dates["order_date"] >= order_customer_dates["signup_date"]
).sum())

### Validation 3 — Customer Signup Date vs Order Date
Business rule: customer signup dae ya to order date se pehle ya same hona chahiye. Agar order date signup date se pehle hai, to ye ek data quality issue hai.

In [ ]:
# return ka status shipment me to delivered hee dikhana chhaiye
returned_items = (
    returns[["order_item_id"]]
    .drop_duplicates()
    .merge(
        order_items[["order_item_id", "order_id"]],
        on="order_item_id",
        how="left"
    )
    .merge(
        shipments[["order_id", "status"]],
        on="order_id",
        how="left"
    )
)
returned_items["status"].value_counts()

In [ ]:
print("Unique returned order items:", returned_items["order_item_id"].nunique())
print("Returned items with delivered shipment:",
      (returned_items["status"] == "delivered").sum())
print("Returned items with non-delivered shipment:",
      (returned_items["status"] != "delivered").sum())

In [ ]:
#5. price of product vs price of ordder item
item_price_check = order_items[
    ["order_item_id", "product_id", "price"]
].merge(
    products[["product_id", "price"]],
    on="product_id",
    how="left",
    suffixes=("_order_item", "_product")
)
item_price_check.head()

In [ ]:
item_price_check["price_order_item"].eq(
    item_price_check["price_product"]
).value_counts()
item_price_check[item_price_check["price_order_item"] == item_price_check["price_product"]].head()

In [ ]:
print("Total order items:", len(item_price_check))
print("Exact price matches:", (
    item_price_check["price_order_item"] ==
    item_price_check["price_product"]
).sum())
print("Price mismatches:", (
    item_price_check["price_order_item"] !=
    item_price_check["price_product"]
).sum())

In [ ]:
# har order ka at least 1 order item hona chahiye, otherwise wo order incomplete hoga
orders_with_items = order_items["order_id"].nunique()
orders_without_items = (
    len(orders) - orders_with_items)
print("Total orders:", len(orders))
print("Orders with order items:", orders_with_items)
print("Orders without order items:", orders_without_items)

# Business Validation Summary

This notebook validates key business relationships and rules across the retail dataset.

## Validations Performed

### 1. Order Completeness
Checked whether every order has at least one order item.

- Total orders: 300,000
- Orders with order items: 259,233
- Orders without order items: 40,767

**Result:** Failed — a significant number of orders have no associated order items.

---

### 2. Order Total vs Payment Amount
Calculated order-level totals from `order_items` using `qty × price` and compared them with `payments.amount`.

- Comparable orders: 259,233
- Exact payment matches: 10
- Payment mismatches: 259,223
- Correlation between calculated order value and payment amount: approximately -0.000428

**Result:** Failed — `payments.amount` cannot be treated as equivalent to the calculated order-item total.

---

### 3. Refund vs Order-Item Value
Aggregated refunds at the `order_item_id` level and compared cumulative refunds with the recorded item value (`qty × price`).

- Unique returned order items: 29,251
- Refunds exceeding item value: 7,498
- Refunds within item value: 21,753
- Average excess refund among anomalies: approximately 1,680.80
- Maximum excess refund: 9,073

**Result:** Failed — a substantial number of returned items have cumulative refunds exceeding their recorded item value.

---

### 4. Customer Signup Date vs Order Date
Checked whether orders occur on or after the customer's recorded signup date.

- Total orders checked: 300,000
- Orders before customer signup: 120,020

**Result:** Failed — the customer signup date cannot be treated as a reliable prerequisite for order activity in this dataset.

---

### 5. Return vs Shipment Status
Checked whether unique returned order items are associated with a delivered shipment.

- Unique returned order items: 29,251
- Returned items with delivered shipment: 9,841
- Returned items with non-delivered shipment: 19,410

**Result:** Failed — a large proportion of returned items are associated with shipments whose status is not `delivered`.

---

### 6. Order-Item Price vs Product Master Price
Compared the recorded transaction price in `order_items` with the corresponding product master price in `products`.

- Total order items: 600,000
- Exact price matches: 137
- Price mismatches: 599,863

**Result:** Failed — `products.price` should not be assumed to represent the actual transaction price recorded in `order_items`.

---

## Overall Conclusion

The dataset contains several material business inconsistencies across payment reconciliation, refunds, customer lifecycle dates, returns/shipments, pricing, and order completeness.

Therefore, downstream analysis should rely only on relationships and fields whose business meaning is sufficiently supported by the data. The identified inconsistencies should be documented as data-quality limitations rather than silently corrected or assumed away.

Basic structural checks such as primary-key uniqueness, foreign-key validity, null/date validity, and invalid numeric values were performed during data profiling and are not repeated here.